### <span id="0">Computer-exam BFVM23DATASC5 - Supervised Learning</span>

# 2024-2025 - first opportunity, Mon. 04 Nov 2025, 12:30-16:30, ZP11/H1.86

Take a minute to read the following instructions and information.

## Materials

On BlackBoard you will find a zip-file containing the following materials:

* this exam `BFVM23DATASC5_T_DataScience5_SupervisedLearning_2425_DSLS_KRPE-LADR.ipynb` (also as `*.pdf`);
* data files:
  * `cheminformatics_data.csv`
  * `codebook.csv`

## Instructions

This is an open book exam that you take on school workstations using your own account. You are allowed to use any materials in your home folder, look up information on the internet, as well as consult any written materials. However, you are **not** allowed to utilize your own devices (e.g. telephone or laptop) or use online communication media or AI-tools (e.g. chat and ChatGPT); this will be considered fraud. You are allowed to leave the room for toilet or coffee breaks, except during the first and last hour of the scheduled exam time. You are not permitted to confer with fellow students about code.

This exam consists of the following parts:

1. **<a href="#1">Part I</a>**

2. **<a href="#2">Part II</a>**

3. **<a href="#3">Part III</a>**

4. **<a href="#4">Part IV</a>**

Each part directs you to perform a particular supervised-machine learning analysis. If you do not understand what is meant, you may ask the exam's supervisor for clarification. Execute the analysis that is requested, and use your own judgement to perform any steps that you deem necessary. **Do not only provide code solutions, but explain by means of comments and/or text markup why you perform steps, as well as what you conclude from the output of an analysis!**

The various parts can be answered separately; if you are unable to (fully) solve one part, an explanation of your intended solution in the form of text or pseudo-code may be awarded partial credit. You can always continue to a next part. Save your notebook regularly to avoid inadvertent loss of your progress!

## Assessment

All parts have the possible number of points to be scored indicated above. Your grade will be calculated as follows:

$$
\text{Grade} = 1 + 9 \cdot \frac {\text{Points Scored}} {\text{Maximum Score}}
$$

* 40 % of your grade is based on the insightfulness of provided explanations, motivations, interpretations, and conclusions drawn from your analyses.

* 40 % of your grade is based on the correctness, completeness, efficiency, and intelligibility of the code(*) that you produce.

* 20 % of your grade is based on whether or not you use your own classes, developed during the course.


Instructions on how to submit your answers after you finish are provided at the very <a href="#X">bottom</a>.

***

## Data

For a series of almost 1700 molecules their activity against a human Glucocorticoid receptor was measures as IC50 concentrations (in nM). This is the concentration at which the compound is still 50% effective, so lower is more potent. 
For each of these molecules 201 features were derived from their chemical structure. In addition, for each molecule a 1024 bit fingerprint was calculated resulting in, in total, 1225 features per molecule.

The dataset contains 3 targets you will be predicting:
1) The IC50 value (`standard_value`)
2) Whether the IC50 value is < 1000 nM (`is_active`)
3) Whether the molecule satisfies Lipinski's "rule of 5" (`lipinski`)

The codebook describes for each column in the data (`variable`) what it contains (`label`), the units (`units`), what type of variable it is (`type`), and a reference where you can find more information about the feature (`reference`). All of these are provided at a "best effort"-basis; the codebook is incomplete and could be wrong in places... Some of the features are easy to interpret, others are more abstract. All fingerprint features are booleans. The feature names are formatted as `fp{idx}` with `idx` running from `0` to `1024` (non inclusive) and are omitted from the codebook.

The `type` can have 4 values:
- nominal: for nominal variables
- ratio: for ratio variables
- bool: for boolean variables
- count: for variables with non-negative integer values

For more information on the features, see the codebook and/or https://deepmol.readthedocs.io/en/stable/deepmol_docs/featurization.html#d-1d-and-2d-descriptors. For more information on the fingerprints, see https://www.rdkit.org/docs/GettingStartedInPython.html#feature-definitions-used-in-the-morgan-fingerprints and/or https://dx.doi.org/10.1021/ci100050t


In [13]:
import pandas as pd
import numpy as np

In [14]:
data = pd.read_csv('cheminformatics_data.csv')
codebook = pd.read_csv('codebook.csv', index_col=False)

In [15]:
data

,molecule_chembl_id,standard_value,lipinski,is_active,canonical_smiles,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,...,fp1014,fp1015,fp1016,fp1017,fp1018,fp1019,fp1020,fp1021,fp1022,fp1023
0,CHEMBL134277,100.0,False,True,CC1=CC(C)(C)Nc2ccc3c(c21)/C(=C/c1ccsc1)Oc1ccc(...,15.158941,15.158941,0.277125,-3.859397,0.481256,...,1,0,0,0,0,0,0,0,0,0
1,CHEMBL415524,77.0,False,True,CCc1ccccc1/C=C1\Oc2ccc(F)cc2-c2ccc3c(c21)C(C)=...,15.415673,15.415673,0.407112,-3.996804,0.469857,...,1,0,0,0,0,0,0,0,0,0
2,CHEMBL336353,320.0,False,True,CC1=CC(C)(C)Nc2ccc3c(c21)/C(=C/c1ccccc1N(C)C)O...,15.467937,15.467937,0.423251,-4.017487,0.474812,...,1,0,0,0,0,0,0,0,0,0
3,CHEMBL413309,250.0,False,True,CC1=CC(C)(C)Nc2ccc3c(c21)/C(=C/c1ccccc1)Oc1c(F...,15.763427,15.763427,0.326528,-3.909144,0.488806,...,1,0,0,0,0,0,0,0,0,0
4,CHEMBL717,10.0,True,True,CC(=O)O[C@]1(C(C)=O)CC[C@H]2[C@@H]3C[C@H](C)C4...,14.561580,14.561580,2.545120,-6.060854,0.647973,...,0,0,0,1,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1684,CHEMBL5282323,6960.0,False,False,COc1cc(C)c(Cl)cc1S(=O)(=O)Nc1ccc2c(c1)CCCN2C(=...,14.344989,14.344989,0.429853,-6.079389,0.535937,...,0,0,0,0,0,0,0,1,0,0
1685,CHEMBL5266323,2070.0,False,False,COc1cc(C)c(Cl)cc1S(=O)(=O)Nc1ccc2c(c1)CCCN2C(=...,14.088936,14.088936,0.350488,-6.056378,0.482584,...,0,0,0,0,0,0,0,0,0,0
1686,CHEMBL5269021,2220.0,False,False,COc1cc(C)c(Cl)cc1S(=O)(=O)Nc1ccc2c(c1)CCCN2C(=...,14.090200,14.090200,0.573348,-6.147867,0.427672,...,0,0,0,0,0,1,0,0,0,0
1687,CHEMBL5272312,3430.0,False,False,COc1ccc(C(=O)N2CCCc3cc(NS(=O)(=O)c4cc(Cl)c(C)c...,14.499154,14.499154,0.521061,-6.162963,0.458223,...,0,0,0,0,0,0,0,0,0,0


In [16]:
codebook

,variable,label,units,type,reference
0,molecule_chembl_id,Molecule identifier,NaN,nominal,NaN
1,standard_value,IC50 activity against Glucocorticoid receptor,nM,ratio,NaN
2,lipinski,Whether the molecule satisfies Lipinski's rule...,NaN,bool,NaN
3,is_active,Whether the standard_value is less than 1000 nM,NaN,bool,NaN
4,canonical_smiles,A SMILES string representing the molecule,NaN,nominal,NaN
...,...,...,...,...,...
201,fr_thiazole,Number of thiazole rings,NaN,count,NaN
202,fr_thiocyan,Number of thiocyanates,NaN,count,NaN
203,fr_thiophene,Number of thiophene rings,NaN,count,NaN
204,fr_unbrch_alkane,Number of unbranched alkanes of at least 4 mem...,NaN,count,NaN


In [17]:
TARGETS = ['standard_value', 'lipinski', 'is_active']
IDENTIFIERS = ['molecule_chembl_id', 'canonical_smiles']
FINGERPRINT_COLUMNS = [f'fp{idx}' for idx in range(1024)]
FEATURE_COLUMNS = [c for c in codebook['variable'] if c not in TARGETS+IDENTIFIERS]

<a id="1" href="#0" style="text-align: right; display: block;">Back to top</a>

## Part I

In this part you will predict the `is_active` target, and see how informative the fingerprint is. **In all following questions, explain why you perform steps, why you picked specific models, as well as what you conclude from the output.**

1) Train and validate a model to predict the `is_active` column using the columns in `FEATURE_COLUMNS` (or a subset thereof)
2) Train and validate a model to predict the `is_active` column using the columns in `FINGERPRINT_COLUMNS` (or a subset thereof)
3) Train and validate a model to predict the `is_active` column using the columns in both `FEATURE_COLUMNS` and `FINGERPRINT_COLUMNS` (or a subset thereof)
4) Is fingerprinting useful?

In [18]:
# YOUR ANSWER GOES HERE
#1
# first thing we need to do is to separate the features from target
X = data[FEATURE_COLUMNS]
y = data['is_active']
from sklearn.model_selection import train_test_split
#we need to split our dataset to train and validation sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
# now we should apply data processing steps, but we should be cautious that we apply fit and transform
# on training data and anly transform on test data
from data_processor import DataProcessor
processor = DataProcessor(k_best_features=10)
processor.fit_transform(X_train,y_train)
processor.transform(X_test)

#now that data is being pre processed and ready for model estimation
# because the is_active is binary and is 0 and 1s then logistic regression can be a good choice
from logisticregressionwithreg import LogisticRegressionWithReg
model = LogisticRegressionWithReg(lambda_=0)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score
score = accuracy_score(y_test,y_pred)
score

C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\__init__.py:811: RuntimeWarning: overflow encountered in square
  X = X**2
C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\feature_selection\_univariate_selection.py:98: RuntimeWarning: overflow encountered in square
  square_of_sums_alldata = sum(sums_args) ** 2
C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\feature_selection\_univariate_selection.py:99: RuntimeWarning: overflow encountered in square
  square_of_sums_args = [s**2 for s in sums_args]
C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\feature_selection\_univariate_selection.py

KeyError: '[(True, True, True, True, False, True, True, True, True, True, False, True, True, True, True, True, True, True, False, True, False, True, True, True, False, True, True, True, False, True, True, True, False, True, True, False, False, True, True, True, True, True, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, False, True, True, False, True, True, True, True, False, True, True, True, False, True, True, True, True, False, False, True, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, True, True, True, True, False, True, True, False, True, True, True, True, True, True, True, False, True, True, False, False, True, True, False, True, True, True, True, False, True, False, True, False, True, False, True, True, False, True, True, True, True, True, False, True, True, True, True, True, False, True, True, False, True, True, True, False, True, True, True, False, True, False, False, Fa

In [ ]:
#2
X = data[FINGERPRINT_COLUMNS]
y = data['is_active']
from sklearn.model_selection import train_test_split
#we need to split our dataset to train and validation sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
# now we should apply data processing steps, but we should be cautious that we apply fit and transform
# on training data and anly transform on test data
from dataprocessor import DataProcessor
processor = DataProcessor(k_best_features=10)
processor.fit_transform(X_train,y_train)
processor.transform(X_test)

#now that data is being pre processed and ready for model estimation
# because the is_active is binary and is 0 and 1s then logistic regression can be a good choice
from logisticregressionwithreg import LogisticRegressionWithReg
model = LogisticRegressionWithReg(lambda_=0)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score
score = accuracy_score(y_test,y_pred)
score

In [ ]:
#3
X = data[FEATURE_COLUMNS+FINGERPRINT_COLUMNS]
y = data['is_active']
from sklearn.model_selection import train_test_split
#we need to split our dataset to train and validation sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
# now we should apply data processing steps, but we should be cautious that we apply fit and transform
# on training data and anly transform on test data
from dataprocessor import DataProcessor
processor = DataProcessor(k_best_features=10)
processor.fit_transform(X_train,y_train)
processor.transform(X_test)

#now that data is being pre processed and ready for model estimation
# because the is_active is binary and is 0 and 1s then logistic regression can be a good choice
from logisticregression import LogisticRegressionWithReg
model = LogisticRegressionWithReg(lambda_=0)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score
score = accuracy_score(y_test,y_pred)
score

In [ ]:
# no fingerprints data are not important and won't add any informative data to our analysis
# because from 0.80 to 0.81 there is difference and them (fingerprints) were 0.83

<a id="2" href="#0" style="text-align: right; display: block;">Back to top</a>

## Part II

In this part you will predict the `lipinski` target. This column is actually derived from 4 columns in `FEATURE_COLUMNS` (`NumHDonors`, `NumHAcceptors`, `MolWt`, and `MolLogP`). Each of these columns was compared to a threshold, and if all 4 features were less than their threshold the molecule satisfies the Lipinski rule-of-5. 
**In all following questions, explain why you perform steps, why you picked specific models, as well as what you conclude from the output.**

1) What kind of model could reproduce this behaviour?
2) Train and validate such a model to find the used threshold values.
3) Train and validate a model to predict the `lipinski` column using the columns in `FEATURE_COLUMNS`, *except* `NumHDonors`, `NumHAcceptors`, `MolWt`, and `MolLogP`. Can you say something about which features are most important now?
3) Train and validate a model to predict the `lipinski` column using the columns in `FINGERPRINT_COLUMNS`

In [19]:
# YOUR ANSWER GOES HERE
#1 the decision tree can reproduce this kind of behaviour as choose best feature and best threshold to 
# split the data into subset right and subset left
#2
X2 = data[['NumHDonors', 'NumHAcceptors', 'MolWt','MolLogP']].values
y2 = data['lipinski']

X2_train,X2_test,y2_train,y2_test = train_test_split(X2,y2,test_size=0.2)

from decisiontree import DecisionTree

dt_clf =DecisionTree()
dt_clf.fit(X2_train,y2_train)
y_pred = dt_clf.predict(X2_test)
the_score = accuracy_score(y2_test,y_pred)
dt_clf.print_tree(depth=4)
# the_score
# from sklearn.tree import DecisionTreeClassifier
# sk_dt_clf = DecisionTreeClassifier()
# sk_dt_clf.fit(X2_train,y2_train)
# sk_y_pred = sk_dt_clf.predict(X2_test)
# the_score_sk = accuracy_score(y2_test,sk_y_pred)
# the_score_sk
X2

ModuleNotFoundError: No module named 'decisiontree'

In [ ]:
# feature 3 means MolLogP should be less than 5.002 and MolWt should be less than 501.38

In [ ]:
def entropy(probs):
    return np.sum(-probs * np.log2(probs + 1e-100))

def determine_split(X, y_onehot):
    """returns the best feature and best split threshold"""
    best_feature = best_threshold = None
    best_entropy = float('inf') 
    for feature in range(X.shape[1]):
        uniques = np.unique(X[:,feature])
        for index in range(len(uniques) -1):
            threshold = uniques[index] + uniques[index + 1] / 2.0
            subset_left = (X[:,feature] <= threshold)
            # subset_right = (X[:,feature] > threshold)
            subset_right = ~subset_left
            
            probs_left = np.mean(y_onehot[subset_left,:],axis=0)
            probs_right = np.mean(y_onehot[subset_right,:],axis=0)
            
            entropy_left = entropy(probs_left)
            entropy_right = entropy(probs_right)
            
            entropy_mean = np.mean(subset_left) * entropy_left \
                            +np.mean(subset_right) * entropy_right
            if entropy_mean < best_entropy:
                best_feature = feature
                best_threshold = threshold
                best_entropy = entropy_mean
                
        return best_feature, best_threshold
import numpy as np    
classes = np.unique(y2_train)
y_onehot = np.array([y2_train]).T == classes
determine_split(X2_train, y_onehot)

Best Feature: 3
Best Threshold: 4.995750000000005


In [ ]:
# 'NumHAcceptors' threshold is 6.5 and 'NumHDonors' is 5.0

In [ ]:
#3 
subtract = ['NumHDonors', 'NumHAcceptors', 'MolWt','MolLogP']
X2_3 = data[FEATURE_COLUMNS]
X2_3 = X2_3.drop(columns = subtract).values
y2_3 = data['lipinski']

X2_3_train,X2_3_test,y2_3_train,y2_3_test = train_test_split(X2_3,y2_3,test_size=0.2)

from decisiontree import DecisionTree

dt_clf =DecisionTree()
dt_clf.fit(X2_3_train,y2_3_train)
y_pred = dt_clf.predict(X2_3_test)
the_score2_3 = accuracy_score(y2_3_test,y_pred)
dt_clf.print_tree(depth=4)

In [ ]:
the_score2_3 

In [ ]:
#4
 
X2_4 = data[FINGERPRINT_COLUMNS].values

y2_4 = data['lipinski']

X2_4_train,X2_4_test,y2_4_train,y2_4_test = train_test_split(X2_4,y2_4,test_size=0.2)

from decisiontree import DecisionTree

dt_clf =DecisionTree()
dt_clf.fit(X2_4_train,y2_4_train)
y_pred = dt_clf.predict(X2_4_test)
the_score2_4 = accuracy_score(y2_4_test,y_pred)
dt_clf.print_tree(depth=4)

In [ ]:
the_score2_4

<a id="3" href="#0" style="text-align: right; display: block;">Back to top</a>

## Part III

Quite a few of the features are counts, which are usually distinctly not normally distributed. Instead, their likelihoods can be modelled using the Gamma distribution (https://en.wikipedia.org/wiki/Gamma_distribution):
$$
f(x;\alpha,\beta) = \frac{x^{\alpha-1} e^{-\beta x} \beta^\alpha}{\Gamma(\alpha)}
$$
$$
\Gamma(\alpha) = (\alpha - 1)! \text{ if } \alpha \in \mathbb{Z} \text{ and } \alpha > 0
$$
Then, if $\mu$ is the mean of your values, and $\sigma^2$ the variance:
$$
\alpha = \frac{\mu^2}{\sigma^2},
\beta = \frac{\mu}{\sigma^2}
$$

This function is implemented as `scipy.stats.gamma` (see https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.gamma.html).
Using only the features with a count (`codebook[codebook['type'] == 'count']['variable']`), predict the `is_active` columns. Compare and contrast the following models:
1) Bernoulli naive Bayes
2) Gaussian naive Bayes
3) Gamma naive Bayes (this one you will need to implement yourself!)

In [ ]:
# YOUR ANSWER GOES HERE
#2 Gaussian naive Bayes
X3 = data[codebook[codebook['type'] == 'count']['variable']].values
y3 = data['is_active']
X3_train,X3_test,y3_train,y3_test = train_test_split(X3,y3,test_size=0.2)

from gaussian_nb import GaussianNaiveBayes

clf = GaussianNaiveBayes()
clf.fit(X3_train,y3_train)
yhat = clf.predict(X3_test)
score = accuracy_score(y3_test,yhat)
score

##sklearn

from sklearn.naive_bayes import GaussianNB
sk_clf = GaussianNB()
sk_clf.fit(X3_train,y3_train)
y_predicted = sk_clf.predict(X3_test)
sk_score = accuracy_score(y3_test,y_predicted)
sk_score

In [ ]:
from bernoulliNB import BernoulliNaiveBayes
#1 bernoulli naive bayes
bnb_clf = BernoulliNaiveBayes()
bnb_clf.fit(X3_train,y3_train)
yhat1 = bnb_clf.predict(X3_test)
score1 = accuracy_score(y3_test,yhat1)
score1

##sklearn
from sklearn.naive_bayes import BernoulliNB
sk2_clf = BernoulliNB()
sk2_clf.fit(X3_train,y3_train)
y2_predicted = sk2_clf.predict(X3_test)
sk2_score = accuracy_score(y3_test,y2_predicted)
sk2_score

In [ ]:
#3
import math
from scipy.stats import gamma
from scipy.special import gamma as gamma_function 
class GammaNaiveBayes:
    def __init__(self):
        self.classes = None
        self.mean = {}
        self.var = {}
        self.priors = {}

    def fit(self, X, y):
        self.classes = np.unique(y)
        for c in self.classes:
            X_c = X[y == c]
            self.mean[c] = X_c.mean(axis=0)
            self.var[c] = X_c.var(axis=0)
            self.priors[c] = X_c.shape[0] / X.shape[0]

    def predict(self, X):
        return [self._predict_single(x) for x in X]

    def _predict_single(self, x):
        posteriors = []
        for c in self.classes:
            prior = np.log(self.priors[c])
            class_conditional = np.sum(np.log(self._pdf(c, x)))
            posterior = prior + class_conditional
            posteriors.append(posterior)
        return self.classes[np.argmax(posteriors)]
    
    def _pdf(self, class_label, x):
        mean = self.mean[class_label]
        var = self.var[class_label]
        alpha = ((mean**2) / var)
        beta = (mean / var)
        numerator = x**(alpha-1) * np.exp(- beta *x) * (beta **alpha)
        denominator = gamma_function(alpha)
        return numerator / denominator
                         
gamma_clf = GammaNaiveBayes()
gamma_clf.fit(X3_train,y3_train)
y_hat = gamma_clf.predict(X3_test)
sre = accuracy_score(y3_test,y_hat)
sre



In [ ]:
#chatgpt

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from scipy.stats import gamma

# Load the data
data = pd.read_csv('cheminformatics_data.csv')
codebook = pd.read_csv('codebook.csv')

# Step 1: Identify "count" features

X = data[codebook[codebook['type'] == 'count']['variable']].values
y = data['is_active'].astype(int)  # Convert to binary (0/1) if not already

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Bernoulli Naive Bayes
# Binarize the data
# Convert to numpy arrays for compatibility with sklearn
X_train_bin = (X_train > 0).astype(int)
X_test_bin = (X_test > 0).astype(int)
y_train = y_train.values
y_test = y_test.values

bernoulli_nb = BernoulliNB()
bernoulli_nb.fit(X_train_bin, y_train)  # Train the model
bernoulli_accuracy = bernoulli_nb.score(X_test_bin, y_test)  # Evaluate on the test set

print(f"Bernoulli NB Accuracy: {bernoulli_accuracy:.2f}")

# Step 4: Gaussian Naive Bayes
gaussian_nb = GaussianNB()
gaussian_nb.fit(X_train, y_train)
gaussian_accuracy = gaussian_nb.score(X_test, y_test)

# Step 5: Gamma Naive Bayes
class GammaNaiveBayes:
    def __init__(self):
        self.classes = None
        self.params = {}
        self.priors = {}

    def fit(self, X, y):
        self.classes = np.unique(y)
        for c in self.classes:
            X_c = X[y == c]
            self.params[c] = {
                'alpha': (X_c.mean(axis=0) ** 2) / X_c.var(axis=0),
                'beta': X_c.mean(axis=0) / X_c.var(axis=0)
            }
            self.priors[c] = len(X_c) / len(X)

    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])

    def _predict_single(self, x):
        posteriors = []
        for c in self.classes:
            prior = np.log(self.priors[c])
            alpha = self.params[c]['alpha']
            beta = self.params[c]['beta']
            likelihood = np.sum(np.log(gamma.pdf(x, a=alpha, scale=1/beta)))
            posterior = prior + likelihood
            posteriors.append(posterior)
        return self.classes[np.argmax(posteriors)]

# Fit Gamma Naive Bayes
gamma_nb = GammaNaiveBayes()
gamma_nb.fit(X_train.values, y_train.values)
y_pred_gamma = gamma_nb.predict(X_test.values)
gamma_accuracy = np.mean(y_pred_gamma == y_test.values)

# Step 6: Compare results
print(f"Bernoulli NB Accuracy: {bernoulli_accuracy:.2f}")
print(f"Gaussian NB Accuracy: {gaussian_accuracy:.2f}")
print(f"Gamma NB Accuracy: {gamma_accuracy:.2f}")


<a id="4" href="#0" style="text-align: right; display: block;">Back to top</a>

## Part IV

One of the holy grails in drug design is the ability to predict the biological activity of a molecule based on only its structure.
**In all following questions, explain why you perform steps, why you picked specific models, as well as what you conclude from the output.**

1) Train and validate the best model you possibly can to predict the `standard_value` column. Use any and all tricks and techniques you can think of.
2) Are you happy with the resulting model? Is it a good model?

In [ ]:
# YOUR ANSWER GOES HERE
from sklearn.model_selection import train_test_split
X4 = data[FEATURE_COLUMNS]
y4 = data['standard_value']
# standard value is not categorical value but numerical value and for predicting it we should use regression like
# linear regression but since we have 201 features it means we have 202 variable that we need to find their value
# so we should use gradient descent for this.
X4_train, X4_test, y4_train, y4_test = train_test_split(X4,y4,test_size=0.2)
from sklearn.preprocessing import StandardScaler
from linearregression import LinearRegressionWithReg
sc = StandardScaler()
X4_trained_scaled = sc.fit_transform(X4_train)
X4_test_scaled = sc.transform(X4_test)
from sklearn.linear_model import LinearRegression
lreg = LinearRegressionWithReg()
lreg.fit(X4_trained_scaled,y4_train)
y_pred = lreg.predict(X4_test_scaled)
from regressionmetrics import RegressionMetrics
r_met = RegressionMetrics(y4_test,y_pred)
mse = r_met.calculate_mse()
r2 = r_met.calculate_r2()
print(r2)
# sk_reg = LinearRegression()
# sk_reg.fit()

-588677324968.7256


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Define a function to evaluate models
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    return {"MSE": mse, "R2": r2, "MAE": mae}


# Initialize Models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# Evaluate Models
results = {}
for name, model in models.items():
    results[name] = evaluate_model(model, X4_trained_scaled, X4_test_scaled, y4_train, y4_test)

# Display Results
for name, metrics in results.items():
    print(f"Model: {name}")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    print()

# Hyperparameter Tuning for Best Model (XGBoost Example)
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5, 7],
}
grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X4_trained_scaled, y4_train)

# Best Model
best_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")
best_metrics = evaluate_model(best_model, X4_trained_scaled, X4_test_scaled, y4_train, y4_test)
print(f"Best Model Evaluation: {best_metrics}")

/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.133e+08, tolerance: 2.400e+05
  model = cd_fast.enet_coordinate_descent(


Model: Linear Regression
  MSE: 21524389570611056.0000
  R2: -10040373475.5529
  MAE: 7980826.3733

Model: Ridge Regression
  MSE: 260110012337144544.0000
  R2: -121332205973.4516
  MAE: 27741598.7199

Model: Lasso Regression
  MSE: 555201159107264000.0000
  R2: -258981885351.1967
  MAE: 40529820.1908

Model: Random Forest
  MSE: 1499454.1914
  R2: 0.3006
  MAE: 587.1517

Model: XGBoost
  MSE: 1488325.5705
  R2: 0.3057
  MAE: 591.1743

Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best Model Evaluation: {'MSE': 1634699.4640492997, 'R2': 0.2374699831959084, 'MAE': 620.2446828663161}


***

<a id="X" href="#0" style="text-align: right; display: block;">Back to top</a>

<div class="alert alert-info">

**Note:** After finishing,

1. evaluate the notebook by means of the menu option `Kernel` > `Restart & Run All` and check that your notebook runs without errors;
2. save the evaluated notebook using the menu option `File` > `Save and Checkpoint`;
3. submit your work by clicking on the assignment title in BlackBoard, attaching all your own work (i.e. this notebook plus any required modules; do *not* include the data file), and submitting it;
4. if in doubt, you may check with the exam supervisor that your submission was successfully received;
5. you are free to leave!
    
*Success!*

</div>